In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Detect project root
PROJECT_ROOT = Path.cwd().parents[0] if "notebooks" in str(Path.cwd()) else Path.cwd()

DATA_FEATURES_DIR = PROJECT_ROOT / "data" / "features"
DATA_LABELS_DIR = PROJECT_ROOT / "data" / "labels"

DATA_FEATURES_DIR, DATA_LABELS_DIR


In [ ]:
features_path = DATA_FEATURES_DIR / "features.csv"

if not features_path.exists():
    raise FileNotFoundError(f"features.csv not found at {features_path}")

df = pd.read_csv(features_path)
df.head()


In [ ]:
# If Time exists, convert it
if "Time" in df.columns:
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    sort_col = "Time"
else:
    # Create a synthetic ordering column
    df["BarIndex"] = np.arange(len(df))
    sort_col = "BarIndex"

df = df.sort_values(sort_col).reset_index(drop=True)
df.head()


In [ ]:
TICK_SIZE = 0.25          # ES tick size
GOOD_SHORT_TICKS = -8     # Profit target for short
BAD_SHORT_TICKS = 5       # Stop-loss threshold
LOOKAHEAD = 10            # Bars to look ahead


In [ ]:
df["FutureClose"] = df["Close"].shift(-LOOKAHEAD)
df["FutureRet"] = (df["FutureClose"] - df["Close"]) / TICK_SIZE

df[["Close", "FutureClose", "FutureRet"]].head(15)


In [ ]:
def label_short(row):
    if pd.isna(row["FutureRet"]):
        return -1  # unknown / ignore
    if row["FutureRet"] <= GOOD_SHORT_TICKS:
        return 1   # good short
    if row["FutureRet"] >= BAD_SHORT_TICKS:
        return 0   # bad short
    return -1      # ambiguous zone → ignore

df["ShortSuccess"] = df.apply(label_short, axis=1)

print("Label counts:")
df["ShortSuccess"].value_counts(dropna=False)


In [ ]:
labeled_df = df[df["ShortSuccess"] != -1].reset_index(drop=True)
labeled_df.head()


In [ ]:
labeled_df["ShortSuccess"].value_counts(normalize=True)


In [24]:
DATA_LABELS_DIR.mkdir(parents=True, exist_ok=True)

out_path = DATA_LABELS_DIR / "labeled.csv"
labeled_df.to_csv(out_path, index=False)

print("Saved to:", out_path)
print("Rows saved:", len(labeled_df))


Saved to: c:\Trading\Projects\ES_AI_Project\data\labels\labeled.csv
Rows saved: 22238


In [ ]:
test = pd.read_csv(out_path)
test.head()


diagnosis of issue 

In [ ]:
df[["Close", "FutureClose", "FutureRet"]].head(20)


In [ ]:
df["ShortSuccess"].value_counts(dropna=False)


In [23]:
len(df[df["ShortSuccess"] != -1])


22238